# Check metadata uncertainty, cutoffs and calibration: deduplicated Subset data

Do the network's scores clearly improve on the spline? Do predicted risks match observed rates, and how much review work follows a cutoff aimed at finding 80% of malignant images?

[07_metadata_tuning.ipynb](07_metadata_tuning.ipynb) now reports tuning for four model families. The preserved fixed-model sections below use verified historical clean NN and spline predictions. [06_metadata_neural_network.ipynb](06_metadata_neural_network.ipynb) reports the current fixed NN comparison, using the same verified clean fixed-NN artifacts. The appended assessment evaluates the new tuned NN separately.

The saved run gives no clear average precision (AP) improvement from the network. The training-selected cutoff finds 80.3% of held-out malignant images while flagging 68.6% of all images. The reserved test labels are not used.

For metric definitions, see the [short concept guide](../concepts.md).


**Deduplicated workflow adaptation.** This notebook uses the frozen `dedup_v1` development manifest and original patient assignments. It reads saved clean artifacts; executing it starts no training and does not load the reserved test. The original notebook is preserved as the presentation template: [08_metadata_robustness.ipynb](../Subset/08_metadata_robustness.ipynb).

The preserved clean fixed-model analysis used **unweighted mean outer-fold AP** as its historical primary measure. Original figure composition and colour meanings are retained. The paired interval plot explicitly displays mean-fold AP, while pooled values remain secondary tables; their values are not interchangeable.

**Scope clarification:** the preserved fixed-model robustness, calibration and decision sections use their verified historical clean artifacts. The additional NN tuning appendix compares the new versioned tuning predictions with the same clean fixed NN and spline baselines. Its findings do not transfer the existing calibration maps or cutoffs to the tuned network.

**Version-2 reporting and evidence boundary:** the current added assessment uses the separately declared ROC-AUC-selected experiment, with mean outer-fold ROC-AUC primary and AP, log loss and Brier secondary. The fixed-model calibration, cutoff, subgroup and AP-led comparisons retained below are historical supporting analyses from the pre-migration report. Their maps and thresholds do not evaluate the new tuned network. The new paired ROC-AUC and secondary AP appendix follows those preserved sections; neither branch fits a final model or opens the reserved test.

## Context and methods

### Separate choosing a rule from assessing it

A fold is a group of patients; all their images stay together. For each of five outer folds:

1. Reserve its patients for assessment.
2. Within the four training folds, fit on three and predict the fourth, four times. These inner out-of-fold (OOF) scores are made without training on the scored patient.
3. Use only inner scores and labels for recalibration and cutoff selection.
4. Fit the spline on all four training folds, predict the outer patients, and assess the chosen rules there. Combine the five outer tables.

Recalibration is a logistic adjustment to the level and spread of probability estimates. Its settings are fixed, rather than chosen from outer results. Calibration asks whether estimates such as 3% agree with observed malignant rates among comparable cases. A score in [0,1] does not establish that agreement.

The cutoff is the score at or above which an image is flagged. Choose the highest inner cutoff meeting 80% recall, the share of malignant images found. This is an academic scenario; outer recall may be lower. Preprocessing is fitted entirely within training, and explicit checks enforce patient separation.

### Read performance and uncertainty together

Average precision (AP) is the primary ranking measure: higher means malignant images concentrate nearer the top. ROC-AUC measures how often a malignant image scores above a benign one; 0.5 is chance. Log loss and Brier score measure probability error; lower is better.

For uncertainty, bootstrap whole patients: repeatedly sample patients with replacement separately within each original outer fold, keeping their images together. The middle 95% of resampled results forms an interval. Compare methods on the same draws to estimate their difference. These intervals keep saved predictions fixed, without retraining.

Results are per image, so patients with more images carry more weight. The reused folds have guided earlier choices; these checks cannot make the comparison an independent final test. The completed clean fits and training-selected cutoffs are already fingerprinted. **Scope qualification:** the original notebook also fitted a final all-development bundle. This report defers that separate clean fit until modelling choices are settled; the historical bundle remains preserved.


### 1. Set up the run

Use the verified historical clean three-run network predictions with `requirements-metadata.txt` and the project Python kernel. The next checks verify their data, code and prediction fingerprints before comparison. Executing these cells reads completed artifacts and writes new reports. The separate runners own model fitting; historical predictions and decisions remain unchanged.


In [ ]:
NOTEBOOK_STAGE = '08_metadata_robustness'
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(ROOT / "src"))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcdefaults()
from IPython.display import display, Markdown, Image
from melanoma_detection.development_dataset import load_development_dataset, load_preserved_inner_memberships
from melanoma_detection.dedup_workflow import load_context, latest_evaluation, load_clean_scores, load_image_report, load_uncertainty_report, read_json, project_path, DIRECTORY, METADATA_SUITE, UNCERTAINTY
get_ipython().run_line_magic("matplotlib", "inline")
MANIFEST = "data/subset/processed/dedup_v1/dataset_manifest.json"
data, dataset_identity = load_development_dataset(ROOT, dataset_manifest=MANIFEST)
context = load_context(ROOT)
from melanoma_detection.metadata_tuning_v2 import condition_directory, DEFAULT_RUN_ID
NN_REPORT = condition_directory(ROOT, "dedup_v1", DEFAULT_RUN_ID) / "report" / NOTEBOOK_STAGE
FIGURE_DIR = NN_REPORT / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def finish_figure(figure, filename, *, dpi=None):
    if not filename or Path(filename).name != filename:
        raise ValueError("Figure name must be a simple stem.")
    figure.savefig(FIGURE_DIR / (filename + ".png"), dpi=dpi or globals().get("ORIGINAL_EXPORT_DPI", 150), bbox_inches="tight")
    figure.savefig(FIGURE_DIR / (filename + ".pdf"), bbox_inches="tight")
    plt.show()
    plt.close(figure)

_score_bundle = None
def score_bundle():
    global _score_bundle
    if _score_bundle is None:
        _score_bundle = load_clean_scores(ROOT)
    return _score_bundle
display(pd.DataFrame([{"Images": len(data), "Malignant images": int(data.target.sum()),
    "Patients": data.patient_id.nunique(), "Malignant patients": data.loc[data.target.eq(1), "patient_id"].nunique()}]))
display(Markdown("**Dataset:** `dedup_v1` · original patient folds · development records only. This notebook reads saved results and starts no training."))

def show_saved_table(relative, limit=30):
    path = project_path(ROOT, relative)
    if not path.exists():
        display(Markdown("**Pending:** " + relative + ". No results are inferred from unfinished runs."))
        return pd.DataFrame()
    table = pd.read_csv(path)
    display(table.head(limit))
    if len(table) > limit:
        display(Markdown(f"Showing {limit} of {len(table)} rows; the complete CSV is saved."))
    return table

def comparison_table():
    evaluation = latest_evaluation(ROOT)
    if "output_directory" not in evaluation:
        display(Markdown("**Pending:** clean evaluation has not been generated."))
        return pd.DataFrame()
    path = project_path(ROOT, evaluation["output_directory"] + "/model_comparison.csv")
    return pd.read_csv(path)

def model_chart(table, title):
    if table.empty:
        display(Markdown("**Pending:** no verified scores are available for this stage."))
        return
    table = table.sort_values("average_precision")
    fig, ax = plt.subplots(figsize=(9, max(3.2, .38 * len(table) + 1.5)), layout="constrained")
    ax.barh(table.model, table.average_precision, color="#276b91")
    ax.set(title=title, xlabel="Unweighted mean outer-fold average precision", xlim=(0, max(.1, float(table.average_precision.max()) * 1.25)))
    finish_figure(fig, "model_comparison_ap")

def pilot_report(run_id):
    directory = ROOT / "results/image_d1_runs" / run_id
    path = directory / "summary.json"
    summary = read_json(path) if path.exists() else {"status": "pending"}
    if summary.get("status") == "complete":
        manifest = read_json(directory / "run_manifest.json")
        if manifest["dataset_identity"] != dataset_identity:
            raise ValueError("Pilot belongs to another dataset.")
        from melanoma_detection.development_dataset import sha256_file
        for relative, digest in summary["artifact_sha256"].items():
            artifact = (directory / relative).resolve()
            if not artifact.is_relative_to(directory.resolve()) or sha256_file(artifact) != digest:
                raise ValueError("Completed pilot artifact changed.")
    return directory, summary


from pathlib import Path
import json
import joblib
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from sklearn.metrics import (
    PrecisionRecallDisplay, RocCurveDisplay, average_precision_score,
    balanced_accuracy_score, brier_score_loss, confusion_matrix, log_loss,
    roc_auc_score, roc_curve, precision_recall_curve,
)
from melanoma_detection.metadata_evaluation import probability_metrics, decision_metrics, calibration_bins
from melanoma_detection.dedup_notebook_metadata import (
    saved_metadata, metric_table, exploratory_cutoff, reliability_table,
    decision_intervals, subgroup_diagnostics, save_metadata_table,
)
from melanoma_detection.dedup_notebook_metadata_templates import marginal_age_profile, metadata_patient_intervals
bundle = score_bundle()
frame, scores = bundle["expected"], bundle["scores"]
meta = saved_metadata(ROOT, bundle)
PROJECT_ROOT = ROOT
train = bundle["frame"].set_index("image_name").loc[frame.image_id].reset_index()
assert np.array_equal(train.patient_id, frame.patient_id)
assert np.array_equal(train.target, frame.target)
assert np.array_equal(train.fold, frame.fold)
assert train.groupby("patient_id").fold.nunique().eq(1).all()
assert train.image_name.is_unique and len(train) == 6706
assert int(train.target.sum()) == 127
FEATURE_COLUMNS = ["sex", "age_approx", "anatom_site_general_challenge"]
FEATURES = FEATURE_COLUMNS
DATA_PATH = ROOT / "data/subset/processed/dedup_v1/dataset_manifest.json"
TRAIN_FOLDS_PATH = DATA_PATH
REPORT_DIR = NN_REPORT / "tables"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURES = FIGURE_DIR
TABLES = METRICS = PREDICTIONS = REPORT_DIR
plt.rcdefaults()

TARGET_RECALL = .80
BOOTSTRAP_REPETITIONS, BOOTSTRAP_SEED = 2000, 42


# Preserve the original notebook's plotting defaults.
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
ORIGINAL_EXPORT_DPI = 180

from melanoma_detection.metadata_roc_historical import (load_clean_scores, latest_evaluation, load_matched_metadata, save_metadata_table)


## Data

### 2. Check inputs and match images

Match predictions by image name and verify patient IDs, labels and folds. Check data, prediction and validation-code fingerprints; the clean manifest and retained-record identity must match both metadata receipts. This prevents comparisons with another dataset or changed method. The table confirms the development population.


In [ ]:

print(f"Manifest: {DATA_PATH.relative_to(ROOT)}")
print(f"Dataset identity: {bundle['identity']['dataset_identity']}")
print(f"Images: {len(train):,}; patients: {train.patient_id.nunique():,}")
print(f"Malignant images: {int(train.target.sum()):,} ({train.target.mean():.2%})")
print(f"Malignant patients: {train.loc[train.target.eq(1), 'patient_id'].nunique():,}")
print("Original patient folds retained; reserved test not loaded.")

nn_predictions = pd.read_csv(meta["base"] / "metadata_nn_oof.csv")
nn_predictions = nn_predictions.set_index("image_name").loc[train.image_name].reset_index()
for column in ["patient_id", "target", "fold"]:
    assert np.array_equal(train[column], nn_predictions[column])
display(pd.DataFrame([{"Images": len(train), "Patients": train.patient_id.nunique(),
    "Malignant images": int(train.target.sum()), "Patients with malignancy": train.loc[train.target.eq(1), "patient_id"].nunique(),
    "Malignant image fraction": train.target.mean()}]).round(4))


## Results

### 3. Verify calibration and cutoffs selected using training patients

The raw spline keeps four cubic age knots and regularisation C=1. For every outer fold, only the inner training scores supply information for recalibration and cutoff selection. The saved audit table records the size and class counts of each partition.

The table below shows each training-selected cutoff beside its results on outer patients. Compare sensitivity (malignant images found), specificity (benign images left unflagged), precision (malignancy among flagged images) and the fraction flagged. These columns show why meeting a recall target can require many false alarms.


In [ ]:

outer_predictions = meta["calibration"].copy()
split_audit = pd.read_csv(meta["base"] / "spline_inner_audit.csv")
assert split_audit.patient_overlap.eq(0).all()
np.testing.assert_allclose(outer_predictions.p_spline_raw, scores["spline_raw"], rtol=0, atol=1e-14)
fold_rows = []
for fold, rows in outer_predictions.groupby("fold"):
    for version, p_column, decision_column, threshold_column in [
        ("Raw spline", "p_spline_raw", "decision_raw", "raw_threshold"),
        ("Recalibrated spline", "p_spline_calibrated", "decision_calibrated", "calibrated_threshold")]:
        fold_rows.append({"fold": fold, "procedure": version, "training_selected_threshold": rows[threshold_column].iloc[0],
            **probability_metrics(rows.target, rows[p_column]), **decision_metrics(rows.target, rows[decision_column])})
fold_metrics = pd.DataFrame(fold_rows)
display(split_audit.round(5))
display(fold_metrics[["fold", "procedure", "training_selected_threshold", "sensitivity", "specificity", "precision", "flagged_fraction"]].round(4))
save_metadata_table(ROOT, NOTEBOOK_STAGE, "training_only_fold_metrics", fold_metrics)


### 4. Read the patient-based comparison intervals

Draw the original number of patients within each outer fold with replacement, 2,000 times (422 patients in total per draw), keeping each person's images together. Compute all methods on each same draw and subtract the spline score. This paired comparison isolates a method difference within the sampled patients.

- **Left panel:** models are the rows and mean-fold AP is horizontal. Blue dots show observed scores; lines show 95% patient-bootstrap intervals. Farther right is better.
- **Right panel:** the horizontal axis is network mean-fold AP minus spline mean-fold AP. Gold squares mark observed differences and lines show their paired intervals. Zero means no change; an interval crossing zero gives no clear direction. Prefer this paired view over comparing the overlap of separate score intervals.

**Aggregation adaptation:** the original figure displayed pooled AP. The clean workflow retains its blue-dot/gold-square layout but displays the primary mean-fold AP. Pooled intervals are also saved as a secondary table. The network mean's interval is [-0.0255, +0.0076], giving no clear AP improvement. These exploratory intervals hold predictions fixed and are not adjusted for multiple comparisons.


In [ ]:

probability_sets = {"Raw spline": scores["spline_raw"], "Recalibrated spline": outer_predictions.p_spline_calibrated.to_numpy(),
    "NN seed 42": scores["nn_seed_42"], "NN seed 142": scores["nn_seed_142"], "NN seed 242": scores["nn_seed_242"],
    "Fixed NN mean of three": scores["metadata_nn_mean"]}
intervals_all = metadata_patient_intervals(frame, probability_sets, BOOTSTRAP_REPETITIONS)
save_metadata_table(ROOT, NOTEBOOK_STAGE, "paired_patient_intervals_all_aggregations", intervals_all)
intervals = intervals_all.query("aggregation == 'mean_fold'").copy()
display(intervals.loc[intervals.metric.isin(["average_precision", "roc_auc"]),
                      ["model_name", "reference_model", "metric", "estimate", "ci_lower", "ci_upper", "n_valid"]].round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
selected = ["Raw spline", "NN seed 42", "Fixed NN mean of three"]
for index, name in enumerate(selected):
    row = intervals[(intervals.model_name == name) & intervals.reference_model.isna() & (intervals.metric == "average_precision")].iloc[0]
    axes[0].plot([row.ci_lower, row.ci_upper], [index, index], color="#2563EB", linewidth=2)
    axes[0].plot(row.estimate, index, "o", color="#1E3A8A")
axes[0].set_yticks(range(len(selected)), selected)
axes[0].set_xlim(left=0)
axes[0].set_xlabel("Mean outer-fold average precision")
axes[0].set_title("Mean-fold AP with patient bootstrap intervals")
for index, name in enumerate(["NN seed 42", "Fixed NN mean of three"]):
    row = intervals[(intervals.model_name == name) & (intervals.reference_model == "Raw spline") & (intervals.metric == "average_precision")].iloc[0]
    axes[1].plot([row.ci_lower, row.ci_upper], [index, index], color="#C58A00", linewidth=2)
    axes[1].plot(row.estimate, index, "s", color="#7C5E10")
axes[1].axvline(0, color="#64748B", linestyle="--")
axes[1].set_yticks([0, 1], ["NN seed 42 - spline", "NN mean - spline"])
axes[1].set_xlabel("Paired change in average precision")
axes[1].set_title("Does the NN improve on the spline?")
for axis in axes: axis.grid(axis="x", alpha=0.2)
fig.suptitle("Development OOF scores: 2,000 whole-patient draws, percentile 95% intervals", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "metadata_patient_bootstrap_d1.png", dpi=180, bbox_inches="tight")
finish_figure(fig, 'metadata_paired_patient_intervals')

### 5. Read calibration and score distributions

Group scores into five roughly equal-sized bins without splitting identical scores. Blue circles are the raw spline; gold squares are its training-only recalibrated version. Axis fractions such as 0.03 mean 3%.

- **Left: calibration.** Horizontal position is mean estimated probability; vertical position is observed malignant share. The dashed diagonal means agreement. Above it indicates underestimated risk; below it indicates overestimated risk. Vertical lines are 95% patient-bootstrap intervals for the observed rate, holding bins fixed. They do not describe uncertainty in horizontal position.
- **Right: score distribution.** Horizontal position is predicted probability; outline height counts images. Blue and gold show how recalibration moves the scores, most of which remain small.

Read the bin counts and error measures too. Few malignant cases make the curve noisy; Brier score and log loss reflect ranking as well as calibration. Recalibration is not automatically an improvement.


In [ ]:

def bin_rate_intervals(analysis_frame, probabilities, bins, repetitions=2000):
    table = reliability_table(analysis_frame, {"current": probabilities}, repetitions=repetitions)
    return table[["bin", "observed_lower", "observed_upper"]]
calibration_tables =  []
fig, axes = plt.subplots(1, 2, figsize=(12, 4.9))
for name, color, marker in [("Raw spline", "#2563EB", "o"), ("Recalibrated spline", "#C58A00", "s")]:
    probabilities = probability_sets[name]
    bins = calibration_bins(train.target, probabilities, n_bins=5, groups=train.patient_id)
    bins = bins.merge(bin_rate_intervals(train, probabilities, bins), on="bin", validate="one_to_one")
    bins.insert(0, "model", name)
    calibration_tables.append(bins)
    axes[0].plot(bins.mean_probability, bins.observed_rate, marker=marker, color=color, label=name)
    axes[0].vlines(bins.mean_probability, bins.observed_lower, bins.observed_upper, color=color, alpha=0.8)
    axes[1].hist(probabilities, bins=np.linspace(0, max(probability_sets["Raw spline"].max(), probability_sets["Recalibrated spline"].max()) + .001, 20), histtype="step", linewidth=2, color=color, label=name)
calibration_table = pd.concat(calibration_tables, ignore_index=True)
calibration_table.to_csv(TABLES / "metadata_calibration_bins_d1.csv", index=False)
limit = max(calibration_table.mean_probability.max(), calibration_table.observed_upper.max()) * 1.15
axes[0].plot([0, limit], [0, limit], linestyle="--", color="#64748B", label="Perfect calibration")
axes[0].set(xlim=(0, limit), ylim=(0, limit), xlabel="Average predicted probability", ylabel="Observed malignant fraction", title="Five-bin calibration with patient intervals")
axes[1].set(xlabel="Predicted probability", ylabel="Number of images", title="Where the probability scores fall")
for axis in axes:
    axis.legend(frameon=False); axis.grid(alpha=0.2)
fig.suptitle("Raw and training-only recalibrated spline: held-out development patients", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "metadata_calibration_d1.png", dpi=180, bbox_inches="tight")
finish_figure(fig, 'metadata_reliability_and_score_distributions')
display(calibration_table.round(4))
display(pd.DataFrame([{"procedure": name, **probability_metrics(train.target, probability_sets[name])}
                      for name in ["Raw spline", "Recalibrated spline"]]).round(5))

### 6. Assess the cutoff trade-off on outer patients

Each outer fold keeps the cutoff selected from its own training data. We combine those decisions without choosing a more favourable cutoff from the outer labels.

In the first table, TP and FN count malignant images found and missed; FP and TN count benign images flagged and left unflagged. Sensitivity is the fraction of malignant images found, specificity is the fraction of benign images left unflagged, and precision is the fraction of flagged images that are malignant. The flagged fraction shows the review workload.

The second table gives 95% patient-bootstrap intervals for those proportions. These keep the saved decisions fixed rather than reselecting a cutoff in each draw. The raw and recalibrated procedures make the same decisions in this run: they find 102 of 127 malignant images and flag 4,597 of 6,706 images.


In [ ]:
decision_rows, decision_ci_tables =  [], []
for name, column in [("Raw spline", "decision_raw"), ("Recalibrated spline", "decision_calibrated")]:
    decision_rows.append({"procedure": name, **decision_metrics(train.target, outer_predictions[column])})
    ci = decision_intervals(train, outer_predictions[column]); ci.insert(0, "procedure", name)
    decision_ci_tables.append(ci)
decision_summary = pd.DataFrame(decision_rows)
decision_ci = pd.concat(decision_ci_tables, ignore_index=True)
decision_summary.to_csv(METRICS / "metadata_nested_threshold_metrics_d1.csv", index=False)
decision_ci.to_csv(METRICS / "metadata_nested_threshold_intervals_d1.csv", index=False)
display(decision_summary.round(4))
display(decision_ci.round(4))

### 7. Read the age, sex and site checks

Each group uses raw spline scores and one common training-selected cutoff procedure. No group-specific cutoff is tuned. Age bands below 50 and 50 or older are fixed reporting groups.

The three panels show age, sex and site. Rows name groups; horizontal position is AP, with larger values meaning better ranking. Blue circles are observed AP and lines are 95% intervals from 1,000 whole-patient draws within the group and its original outer-fold strata. Labels count patients with a malignant image. Crosses mark fewer than ten such patients, with intervals suppressed; `AP undefined` means no malignant images to assess ranking.

Check counts, malignant share and sensitivity in the table before comparing groups. AP depends partly on how common malignancy is, so it cannot establish fairness alone. Sparse groups and wide intervals limit conclusions, and one patient can appear in multiple age or site groups.


In [ ]:

subgroup_table = subgroup_diagnostics(train, outer_predictions.p_spline_raw.to_numpy(),
    outer_predictions.decision_raw.to_numpy(), repetitions=1000)
subgroup_table = subgroup_table.loc[subgroup_table.field.ne("Site availability")].copy()
subgroup_table = subgroup_table.rename(columns={"malignant_images": "positive_images", "malignant_patients": "positive_patients"})
subgroup_table["sensitivity_lower"], subgroup_table["sensitivity_upper"] = np.nan, np.nan
ages = pd.to_numeric(train.age_approx, errors="coerce")
report_groups = {"Age band": pd.Series(np.where(ages.isna(), "Missing", np.where(ages < 50, "<50", ">=50")), index=train.index),
    "Sex": train.sex.fillna("Missing").astype(str), "Site": train.anatom_site_general_challenge.fillna("Missing").astype(str)}
for index, row in subgroup_table.iterrows():
    if row.positive_patients >= 10:
        mask = report_groups[row.field].eq(row.group).to_numpy()
        subset = train.loc[mask].reset_index(drop=True)
        ci = decision_intervals(subset, outer_predictions.loc[mask, "decision_raw"].to_numpy(), repetitions=1000)
        interval = ci.query("metric == 'sensitivity'").iloc[0]
        subgroup_table.loc[index, ["sensitivity_lower", "sensitivity_upper"]] = [interval.ci_lower, interval.ci_upper]
save_metadata_table(ROOT, NOTEBOOK_STAGE, "subgroup_checks", subgroup_table)
display(subgroup_table[["field", "group", "images", "patients", "positive_images", "positive_patients", "prevalence",
    "average_precision", "average_precision_lower", "average_precision_upper", "sensitivity", "evidence_note"]].round(4))
fig, axes =  plt.subplots(1, 3, figsize=(15.5, 5.8), sharex=True)
subgroup_maximum = max(subgroup_table.average_precision.max(), subgroup_table.average_precision_upper.max()) * 1.12
for axis, field in zip(axes, report_groups):
    rows = subgroup_table[subgroup_table.field.eq(field)].reset_index(drop=True)
    for index, row in rows.iterrows():
        if pd.notna(row.average_precision_lower):
            axis.plot([row.average_precision_lower, row.average_precision_upper], [index, index], color="#2563EB", linewidth=2)
        axis.plot(row.average_precision, index, "o" if row.positive_patients >= 10 else "x", color="#1E3A8A")
        if pd.isna(row.average_precision):
            axis.text(subgroup_maximum * .05, index, "AP undefined", va="center", fontsize=9, color="#64748B")
    axis.set_yticks(range(len(rows)), [f"{row.group} ({row.positive_patients} positive patients)" for row in rows.itertuples()])
    axis.set_title(field); axis.set_xlabel("Average precision"); axis.set_xlim(0, subgroup_maximum); axis.grid(axis="x", alpha=0.2)
fig.suptitle("Subgroup development checks: 95% patient intervals; x = sparse group without interval", fontsize=11)
fig.tight_layout()
fig.savefig(FIGURES / "metadata_subgroups_d1.png", dpi=180, bbox_inches="tight")
finish_figure(fig, 'metadata_subgroup_support_and_intervals')

### 8. Prepare the retained model for later final fitting and testing

Keep the previously chosen raw four-knot spline as the development comparator. The original stage fitted it on all development patients and selected its academic cutoff from five-fold development OOF scores, then saved the model, cutoff, training identities, source fingerprints and dependency versions. That combination gives a later test script a complete fixed recipe.

**Experiment-scope change:** this clean report deliberately defers all-development fitting until the final procedure is chosen. It reads the five clean outer models and their training-selected calibration/cutoff artifacts; these support development evaluation, rather than providing a final deployable model. The historical `models/metadata_d1` bundle belongs to the original 6,864-image data and cannot be relabelled as deduplicated.

After settling features, model settings, calibration and cutoff rules, fit a separately named clean bundle on all 6,706 development records, preserving patient identities and exact deduplication fingerprints. Verify reserved-test patient/image separation and the consistent duplicate policy before scoring. Evaluate once with the saved procedure, without adjusting it to improve the test result. This notebook performs neither that fit nor the test.

In [ ]:

display(pd.DataFrame([{"chosen_development_comparator": "Raw four-knot spline; C=1",
    "fitted_clean_final_model": False, "reserved_test_loaded": False,
    "historical_final_bundle": "models/metadata_d1/frozen_spline.joblib",
    "next_required_step": "Settle the complete recipe, fit a separate clean all-development bundle, then test once"}]))
print("No final fit or final-test evaluation is performed by this report.")


### 9. Check the results together

The printed recap below brings together the AP comparison, its uncertainty interval, the cutoff's detection and review workload, and the number of sparse reporting groups. The recap uses primary mean-fold AP and its matching paired interval, alongside pooled image-level decision rates. Read these as development findings for the retained spline.


In [ ]:

recap = metric_table(frame, {"spline_raw": probability_sets["Raw spline"], "metadata_nn_mean": probability_sets["Fixed NN mean of three"]},
    ["spline_raw", "metadata_nn_mean"]).query("scope == 'mean_fold'").set_index("model")
raw_ap, nn_ap = recap.loc["spline_raw", "average_precision"], recap.loc["metadata_nn_mean", "average_precision"]
delta = intervals[(intervals.model_name == "Fixed NN mean of three") & (intervals.reference_model == "Raw spline") & intervals.metric.eq("average_precision")].iloc[0]
raw_decisions = decision_summary.query("procedure == 'Raw spline'").iloc[0]
print(f"Primary mean-fold AP: raw spline {raw_ap:.4f}; mean-of-three NN {nn_ap:.4f}.")
print(f"NN minus spline mean-fold AP: {delta.estimate:+.4f}, patient 95% interval [{delta.ci_lower:+.4f}, {delta.ci_upper:+.4f}].")
print(f"Training-target recall: {TARGET_RECALL:.0%}; held-out raw-spline sensitivity: {raw_decisions.sensitivity:.1%}.")
print(f"At that policy, {raw_decisions.flagged_fraction:.1%} of images are flagged; precision is {raw_decisions.precision:.1%}.")
print(f"Sparse reporting groups: {int(subgroup_table.positive_patients.lt(10).sum())} of {len(subgroup_table)}.")
print("Primary development comparator remains the raw spline. Final clean fit is deferred; reserved test remains closed.")


## Limitations

These folds have already guided model comparisons. Keeping calibration and cutoff selection within training prevents direct outer-label reuse for those steps, but cannot undo earlier model exploration. Independent final testing is still needed after decisions are fixed.

Bootstrap intervals hold predictions and decisions fixed. They cover patient-sampling variation, without including retraining, method selection or a new hospital. Multiple exploratory comparisons are not adjusted for repeated testing.

Only 127 images are malignant; three of eleven reporting groups have fewer than ten patients with a malignant image. AP changes with malignant prevalence, and sparse subgroup results cannot establish fairness or causation. Five calibration bins can hide within-bin differences.

The 80% recall target is an academic scenario. Flagging roughly 69% of images does not establish acceptable clinical workload or safety. The metadata model predicts image outcomes without seeing lesion appearance.


## Summary

The raw spline has primary mean-fold AP 0.0648; the three-network mean has 0.0596. Their difference is -0.0053 with a 95% paired patient interval of [-0.0255, +0.0076], giving no clear network improvement. The original-style pooled AP values are 0.0492 and 0.0477. Recalibration slightly lowers probability error; within-fold ranking is unchanged by its monotonic mapping, although pooled ranking shifts because folds use different mappings.

Keep the raw spline as the predeclared development comparator. The training-selected 80% recall scenario finds 102 of 127 malignant images (80.3%), flags 4,597 of 6,706 images (68.6%) and has 2.2% precision. Raw and recalibrated procedures make the same decisions here. That workload limits metadata alone, and three of eleven age/sex/site reporting groups have fewer than ten malignant patients.

**Final clean fitting remains deferred.** Existing outer models and cutoffs are saved for development assessment; the historical original-data final bundle remains preserved. The reserved test has not been evaluated. Next, [09_image_modelling.ipynb](09_image_modelling.ipynb) assesses clean image information. Complete and freeze the modelling procedure before fitting a separate clean final bundle and evaluating the reserved test once.

**Historical fixed-model summary:** read the separately declared version-2 assessment below for the current ROC-AUC-first model comparison.

## Additional deduplicated-workflow diagnostics

The complete original section sequence above is retained as the template. The following clearly labelled additions document clean artifact verification, training support and explanatory diagnostics that were not figures in the original. They remain secondary development evidence and do not select another model or cutoff.

In [ ]:

import json
from melanoma_detection.metadata_evaluation import decision_metrics
from melanoma_detection.dedup_notebook_metadata import (
    LABELS, COLORS, saved_metadata, metric_table, curve_figure, metric_bar_figure,
    cutoff_table, confusion_figure, estimator_diagnostics, reliability_table,
    decision_intervals, subgroup_diagnostics, save_metadata_table,
)
bundle = score_bundle()
frame, scores = bundle["expected"], bundle["scores"]
meta = saved_metadata(ROOT, bundle)
display(pd.DataFrame([{"Dataset identity": bundle["identity"]["dataset_identity"],
    "New CPU fits": meta["receipt"]["executed_fits"], "Reused clean fits": meta["receipt"]["reused_fits"],
    "Images with matching predictions": len(frame), "Reserved test loaded": False}]))


In [ ]:
calibration = meta["calibration"]

In [ ]:
COLORS = ['#2563EB', '#C58A00', '#6B7F2A', '#EA580C', '#303030', '#94A3B8']

### Added diagnostic: 3. Compare probability error and equal-patient weighting

Raw versus recalibrated score error must be assessed on outer patients. Equal-patient image weights 1/(retained images for that patient) provide a secondary diagnostic for unequal patient contribution; they do not convert the task into patient cancer diagnosis or change the primary metric.

In [ ]:
probabilities = {"spline_raw": calibration.p_spline_raw.to_numpy(), "spline_calibrated": calibration.p_spline_calibrated.to_numpy(), "metadata_nn_mean": scores["metadata_nn_mean"]}
display(metric_table(frame, probabilities, list(probabilities)).query("scope != 'fold'").round(5))
from sklearn.metrics import average_precision_score, roc_auc_score
weights = 1 / frame.groupby("patient_id").patient_id.transform("size").to_numpy()
display(pd.DataFrame([{"model": LABELS[name], "pooled_image_AP": average_precision_score(frame.target, values), "equal_patient_AP": average_precision_score(frame.target, values, sample_weight=weights), "pooled_image_AUC": roc_auc_score(frame.target, values), "equal_patient_AUC": roc_auc_score(frame.target, values, sample_weight=weights)} for name, values in probabilities.items()]).round(5))

### Added diagnostic: 5. Assess frozen cutoffs on outer patients

Each outer fold uses its own training-selected cutoff. Combine its saved decisions without choosing another cutoff from outer labels. Sensitivity counts malignant images found; specificity counts benign images left unflagged; precision is malignant share among flags; flagged fraction estimates review workload. These are image-level decisions with patient-cluster uncertainty.

In [ ]:
decision_rows, interval_rows = [], []
for scale, name in [("raw", "spline_raw"), ("calibrated", "spline_calibrated")]:
    decisions = calibration[f"decision_{scale}"].to_numpy()
    decision_rows.append({"procedure": LABELS[name], **decision_metrics(frame.target, decisions)})
    ci = decision_intervals(frame, decisions)
    ci.insert(0, "procedure", LABELS[name]); interval_rows.append(ci)
decision_summary = pd.DataFrame(decision_rows)
decision_ci = pd.concat(interval_rows, ignore_index=True)
display(decision_summary.round(5)); display(decision_ci.round(5))
save_metadata_table(ROOT, "08_metadata_robustness", "training_only_decision_summary", decision_summary)
save_metadata_table(ROOT, "08_metadata_robustness", "training_only_decision_intervals", decision_ci)
scenarios = [(LABELS[name] + " · training-only cutoff", calibration[column].to_numpy()) for name, column in [("spline_raw", "decision_raw"), ("spline_calibrated", "decision_calibrated")]]
finish_figure(confusion_figure(frame.target, scenarios, "Training-selected metadata decision support"), "metadata_training_only_confusions")

### Added diagnostic: 6. Read detection and review-workload intervals

The same 2,000 patient draws assess raw and recalibrated frozen decisions. Intervals do not reselect cutoffs in each draw or guarantee a clinical workload. Undefined rates retain missing values, and valid-draw counts show whether scarce classes limit estimation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), layout="constrained")
for ax, metric, label in zip(axes, ["sensitivity", "flagged_fraction"], ["Malignant images found (recall)", "All images flagged (review fraction)"]):
    rows = decision_ci.loc[decision_ci.metric.eq(metric)].reset_index(drop=True)
    for index, row in rows.iterrows():
        ax.plot([row.ci_lower, row.ci_upper], [index, index], color=COLORS[index], linewidth=3)
        ax.plot(row.estimate, index, "o", color=COLORS[index])
    ax.set(yticks=range(len(rows)), yticklabels=rows.procedure, xlabel=label, title=label + " with patient intervals", xlim=(0, 1))
    ax.grid(axis="x", alpha=.2)
axes[0].axvline(.8, color="#64748b", linestyle="--", label="Training target")
axes[0].legend(frameon=False)
finish_figure(fig, "metadata_detection_and_review_intervals")

### Added diagnostic: 7. Check age, sex, site and site availability with support

Use raw spline scores and the same frozen decision procedure for every subgroup; no subgroup-specific cutoff is fitted. AP depends partly on malignant prevalence. One patient's images may occupy multiple age/site groups, so counts cannot be summed as independent patients. Secondary intervals use 2,000 seed-42 whole-patient draws within outer folds. Fewer than ten malignant patients triggers interval suppression as a reporting caution, not a zero metric; undefined values remain missing.

In [ ]:
diagnostic_frame = bundle["frame"].set_index("image_name").loc[frame.image_id].reset_index()
subgroups = subgroup_diagnostics(diagnostic_frame, calibration.p_spline_raw.to_numpy(), calibration.decision_raw.to_numpy())
save_metadata_table(ROOT, "08_metadata_robustness", "subgroup_diagnostics_with_patient_support", subgroups)
display(subgroups[["field", "group", "images", "patients", "malignant_images", "malignant_patients", "prevalence", "average_precision", "roc_auc", "sensitivity", "precision", "flagged_fraction", "valid_draws", "evidence_note"]].round(5))
fig, axes = plt.subplots(1, 4, figsize=(18, 6), layout="constrained")
limit = max(float(subgroups.average_precision.max()), float(subgroups.average_precision_upper.max())) * 1.15
for ax, field in zip(axes, ["Age band", "Sex", "Site", "Site availability"]):
    rows = subgroups.loc[subgroups.field.eq(field)].reset_index(drop=True)
    for index, row in rows.iterrows():
        if pd.notna(row.average_precision_lower): ax.plot([row.average_precision_lower, row.average_precision_upper], [index, index], color="#2563eb", linewidth=2)
        if pd.notna(row.average_precision): ax.plot(row.average_precision, index, "o" if row.malignant_patients >= 10 else "x", color="#2563eb")
        else: ax.text(limit * .04, index, "AP undefined", va="center", fontsize=9)
    ax.set(yticks=range(len(rows)), yticklabels=[f"{row.group}\n({row.malignant_patients} malignant patients)" for row in rows.itertuples()], xlabel="Pooled subgroup AP", title=field, xlim=(0, limit))
    ax.tick_params(axis="y", labelsize=9); ax.grid(axis="x", alpha=.2)
finish_figure(fig, "metadata_subgroup_support_and_intervals")

### Additional ROC-AUC-selected metadata experiment

The new experiment selects spline, Random Forest, XGBoost and NN settings by unweighted mean inner-fold ROC-AUC, with preprocessing and duration selection confined to outer-training patients. The completed earlier experiment remains AP-selected history; it is not retrospectively described as ROC-AUC-tuned. The new three-seed network still trains with binary cross-entropy and selects its duration using stopping loss. Its probabilities are averaged before scoring.

Mean outer-fold ROC-AUC is the primary model comparison. Mean outer-fold AP, log loss and Brier score are secondary; pooled metrics are supporting summaries. These are image-level metrics within patient-held-out folds, not patient-level diagnoses. Original results retain duplicate-bearing development records; the deduplicated condition supplies the main clean-data evidence.

The paired comparisons use 2,000 whole-patient resamples within each preserved outer fold. Every model receives the same draw. The intervals condition on saved predictions and do not include repeating the complete fitting and selection process. An interval containing zero leaves the difference unresolved and does not establish equivalence. An observed leading score does not prove superiority.

This is an additional development experiment after earlier outer-fold findings were examined. The revised objective and search were frozen before execution, without expanding the grid in response to new outer scores. Historical calibration maps and recall cutoffs remain attached to their original fixed procedures; they cannot be transferred to the newly ROC-selected NN or spline. This appendix does not replace the final model, fit an all-development model or open the reserved test.


In [ ]:
from melanoma_detection.metadata_tuning_v2_evaluation import show_assessment
import matplotlib.pyplot as plt
plt.rcdefaults()
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
show_assessment(ROOT, 'dedup_v1', run_id='roc_auc_grid36_v1', experiment_version=2)